<a href="https://colab.research.google.com/github/dxda6216/SPARK_fluorescence_anisotropy/blob/main/SPARK_fluorescence_anisotropy_data_pre_processing_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Pre-process SPARK fluorescence anisotropy data
#@markdown **Pre-processes a fluorescence anisotropy data file (Excel) exported by a Tecan SPARK multimode microplate reader.**

#@markdown This only works with the specific Excel layout described on the [GitHub repository page](https://github.com/dxda6216/SPARK_fluorescence_anisotropy).

#@markdown 1. Enter the G-factor below.
#@markdown 2. **Runtime** → **Restart and run all** (or press **Ctrl+M**, then **Ctrl+F9**).
#@markdown 3. When a `Choose Files` / `Browse...` button appears below, click it and select your SPARK Excel file.
#@markdown 4. Wait a minute or two. A CSV file, two Excel files, a ZIP file and (optionally) a PDF of plots will be saved to your computer's Downloads folder.

G_factor = 1.000  #@param {type:"number"}
Plotting_data = "Yes"  #@param ["Yes", "No"]

import glob
import math
import os
import re
import zipfile
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from google.colab import files

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

# Text that marks the start of each data block in the SPARK export,
# mapped to the sheet name used for that block in the "_SPLITTED" workbook.
SECTIONS = {
    'Label 1 [mP]':              'Polarization (mP)',
    'Rawdata (perpendicular)':   'Rawdata Perpendicular',
    'Rawdata (parallel)':        'Rawdata Parallel',
    'Anisotropy':                'Anisotropy',
    'Total Intensity':           'Total Intensity',
    'Intensity (parallel)':      'Intensity Parallel',
    'Intensity (perpendicular)': 'Intensity Perpendicular',
}

# Every output table starts with these columns; the well columns follow.
TIME_COLUMNS = ['Cycle Nr.', 'DateTime', 'Time [hours]', 'Time [s]',
                'Time [m]', 'Time [h]', 'Time [days]', 'Temp. [°C]']

DATETIME_PATTERN = re.compile(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}')

PLOT_RC = {
    'font.size': 5,
    'axes.titlesize': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 5,
    'figure.titlesize': 8,
}


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def scan_csv(csv_path):
    """Return the 1-based line number of each section label and the run start time."""
    with open(csv_path, encoding='utf-8') as f:
        lines = f.read().splitlines()

    label_lines = {}
    start_dt = None
    for line_no, line in enumerate(lines, start=1):
        for label in SECTIONS:
            if label in line:
                label_lines[label] = line_no  # the last occurrence wins
        if 'Start Time,,,,' in line:
            match = DATETIME_PATTERN.search(line)
            if match:
                start_dt = datetime.strptime(match.group(), '%Y-%m-%d %H:%M:%S')

    missing = [label for label in SECTIONS if label not in label_lines]
    if missing:
        raise ValueError(f'Section(s) not found in the data file: {missing}')
    if start_dt is None:
        raise ValueError('"Start Time" not found in the data file.')
    return label_lines, start_dt


def read_block(csv_path, label_line, n_rows):
    """Read one data block: the header row right below the label, then n_rows rows."""
    block = pd.read_csv(csv_path, skiprows=label_line, nrows=n_rows, index_col=False)
    expected = ['Cycle Nr.', 'Time [s]', 'Temp. [°C]']
    if list(block.columns[:3]) != expected:
        raise ValueError(f'Unexpected columns below line {label_line}: '
                         f'{list(block.columns[:3])} (expected {expected})')
    return block


def make_time_table(block, start_dt):
    """Cycle number, time in several units, and temperature for each time point."""
    t = block['Time [s]']
    return pd.DataFrame({
        'Cycle Nr.':    block['Cycle Nr.'],
        'DateTime':     start_dt + pd.to_timedelta(t.astype(int), unit='s'),
        'Time [hours]': t / 3600,
        'Time [s]':     t,
        'Time [m]':     t / 60,
        'Time [h]':     t / 3600,
        'Time [days]':  t / 86400,
        'Temp. [°C]':   block['Temp. [°C]'],
    })


def add_time_columns(block, times):
    """Insert the extra time columns so the block's columns start with TIME_COLUMNS."""
    block = block.copy()
    for position, column in [(1, 'DateTime'), (2, 'Time [hours]'), (4, 'Time [m]'),
                             (5, 'Time [h]'), (6, 'Time [days]')]:
        block.insert(position, column, times[column])
    return block


def with_times(times, values):
    return pd.concat([times, values], axis=1)


def dat_filename(well):
    """'A1' -> 'A01.dat', 'A10' -> 'A10.dat' (zero-padded for LumiCycle Analysis)."""
    match = re.fullmatch(r'([A-Za-z]+)(\d+)', well)
    if match is None:
        return f'{well}.dat'
    return f'{match.group(1)}{int(match.group(2)):02d}.dat'


def plot_wells(pdf_path, hours, wells, panels):
    """One page per well; each panel is (table, y label, title prefix)."""
    x_min = math.floor(hours.min() / 24) * 24
    x_max = math.ceil(hours.max() / 12) * 12 + 6
    x_ticks = list(range(x_min, x_max, 12))

    with plt.rc_context(PLOT_RC), PdfPages(pdf_path) as pdf:
        for well in wells:
            fig, axes = plt.subplots(len(panels), 1, figsize=(8.5, 11))
            fig.subplots_adjust(hspace=0.5)
            for ax, (table, ylabel, title) in zip(axes, panels):
                ax.plot(hours, table[well])
                ax.set_xlim(x_min - 6, x_max)
                ax.set_xticks(x_ticks)
                ax.set_xlabel('Time (hours)')
                ax.set_ylabel(ylabel)
                ax.set_title(f'{title} : Well {well}')
                ax.grid(True)
            pdf.savefig(fig)
            plt.close(fig)


# ---------------------------------------------------------------------------
# 1. Upload the SPARK Excel file
# ---------------------------------------------------------------------------

for pattern in ('*.xlsx', '*.csv', '*.dat', '*.zip', '*.pdf'):
    for old_file in glob.glob(pattern):
        os.remove(old_file)

uploaded = files.upload()
if not uploaded:
    raise SystemExit('No file was uploaded.')

start_time = datetime.now(timezone.utc)
processed_dnt_str = start_time.strftime('%Y-%m-%d %H:%M:%S')
print(f'\nStarted at {processed_dnt_str} (UTC)')

spark_excel_filename = next(iter(uploaded))
print(f'\n"{spark_excel_filename}" has been uploaded.')

base_name = os.path.splitext(spark_excel_filename)[0]
make_plots = Plotting_data == 'Yes'
output_files = {
    'CSV':   base_name + '.csv',
    'Excel': base_name + '_SPLITTED.xlsx',
    'Excel (time series)': base_name + '_Anisotropy_TS.xlsx',
    'ZIP':   base_name + '_for_LCA.zip',
}
if make_plots:
    output_files['PDF'] = base_name + '_Plots.pdf'

print('\nThe following files will be saved:')
for kind, filename in output_files.items():
    print(f'  {kind}: "{filename}"')

# ---------------------------------------------------------------------------
# 2. Convert to CSV and split it into data blocks
# ---------------------------------------------------------------------------

print('\nReading the Excel file and generating a CSV file...')
spark_sheet = pd.read_excel(spark_excel_filename, header=None, index_col=False)
spark_sheet.to_csv(output_files['CSV'], index=False, header=False)

print('\nChecking the structure of the data file...')
label_lines, start_dt = scan_csv(output_files['CSV'])
first_line = label_lines['Label 1 [mP]']
n_time_points = label_lines['Rawdata (perpendicular)'] - first_line - 3

print('\nGenerating Pandas dataframes...')
raw_blocks = {}
for label in SECTIONS:
    line_no = label_lines[label]
    print(f'  {label}: lines {line_no}-{line_no + n_time_points}')
    raw_blocks[label] = read_block(output_files['CSV'], line_no, n_time_points)

times = make_time_table(raw_blocks['Label 1 [mP]'], start_dt)
blocks = {label: add_time_columns(block, times) for label, block in raw_blocks.items()}

# Keep only the columns that actually contain data (judged from the mP block).
print('\nExtracting columns containing data...')
valid_columns = blocks['Label 1 [mP]'].dropna(axis=1, how='all').columns.tolist()
wells = valid_columns[len(TIME_COLUMNS):]
blocks = {label: block[valid_columns] for label, block in blocks.items()}

# ---------------------------------------------------------------------------
# 3. Polarization and anisotropy corrected by the G-factor
# ---------------------------------------------------------------------------

print('\nCalculating polarization and anisotropy with G-factor...')
g = G_factor
# Saturated readings ("OVER") become NaN and are left blank in the outputs.
i_para = blocks['Rawdata (parallel)'][wells].apply(pd.to_numeric, errors='coerce')
i_perp = blocks['Rawdata (perpendicular)'][wells].apply(pd.to_numeric, errors='coerce')

polarization = with_times(times, 1000 * ((i_para - i_perp) / (i_para + g * i_perp)))
anisotropy = with_times(times, 1000 * ((i_para - g * i_perp) / (i_para + 2 * g * i_perp)))
s_values = with_times(times, i_para + 2 * g * i_perp)
d_values = with_times(times, i_para - g * i_perp)

total_time = times['Time [s]'].iloc[-1] - times['Time [s]'].iloc[0]
time_interval = round(total_time / 3600 / max(len(times) - 1, 1), 12)
print(f'Number of rows: {len(times)}')
print(f'First time point: {times["Time [s]"].iloc[0]} sec')
print(f'Last time point: {times["Time [s]"].iloc[-1]} sec')
print(f'Total time duration: {total_time} sec = {total_time / 60} min = '
      f'{total_time / 3600} hours = {total_time / 86400} days')
print(f'Time interval: {time_interval} h')

with pd.ExcelWriter(output_files['Excel (time series)']) as writer:
    anisotropy[['Time [h]', 'Temp. [°C]'] + wells].to_excel(
        writer, sheet_name=f'Every {time_interval:.12f} h', index=False)

# ---------------------------------------------------------------------------
# 4. Plots (one PDF page per well)
# ---------------------------------------------------------------------------

if make_plots:
    print('\nPlotting...')
    plot_wells(output_files['PDF'], times['Time [hours]'], wells, [
        (i_para,       'Fluorescence\n(parallel)',        'Fluorescence (parallel)'),
        (i_perp,       'Fluorescence\n(perpendicular)',   'Fluorescence (perpendicular)'),
        (s_values,     'Ipara + 2 x G x Iperp',           'Ipara + 2 × G × Iperp'),
        (d_values,     'Ipara - G x Iperp',               'Ipara - G × Iperp'),
        (anisotropy,   'Fluorescence\nAnisotropy (mA)',   'Anisotropy'),
    ])

# ---------------------------------------------------------------------------
# 5. Multi-sheet Excel file with all pre-processed data
# ---------------------------------------------------------------------------

print('\nPreparing a new Excel file to save pre-processed data...')
print('It will take 2-3 min. Wait a while, please.')

note = pd.DataFrame({
    'A': ['Original Data Filename', 'Processed Date and Time (UTC)', 'G-factor', 'Sample Wells'],
    'B': '', 'C': '', 'D': '',
    'E': [spark_excel_filename, processed_dnt_str, g, ', '.join(wells)],
})

with pd.ExcelWriter(output_files['Excel']) as writer:
    spark_sheet.to_excel(writer, sheet_name='Spark Datasheet', index=False, header=False)
    note.to_excel(writer, sheet_name='Note', index=False, header=False)
    times.to_excel(writer, sheet_name='Time and Temp')
    for label, sheet_name in SECTIONS.items():
        blocks[label].to_excel(writer, sheet_name=sheet_name)
    polarization.to_excel(writer, sheet_name='Polarization corr by G factor')
    anisotropy.to_excel(writer, sheet_name='Anisotropy corr by G factor')
    s_values.to_excel(writer, sheet_name='S values corr by G factor')
    d_values.to_excel(writer, sheet_name='D values corr by G factor')

# ---------------------------------------------------------------------------
# 6. One anisotropy .dat file per well for LumiCycle Analysis, zipped
# ---------------------------------------------------------------------------

print('\nPreparing an anisotropy data file (readable with LumiCycle Analysis) for each well...')
with zipfile.ZipFile(output_files['ZIP'], 'w', zipfile.ZIP_DEFLATED) as zf:
    for well in wells:
        dat_file = dat_filename(well)
        anisotropy.to_csv(dat_file, header=False, index=False, sep='\t',
                          columns=['Time [days]', well])
        zf.write(dat_file)
        print(f'  {dat_file}')

# ---------------------------------------------------------------------------
# 7. Download everything
# ---------------------------------------------------------------------------

print('\nDownloading the output files into the "Downloads" folder on your computer...')
for filename in output_files.values():
    files.download(filename)

end_time = datetime.now(timezone.utc)
print(f'\nElapsed time: {(end_time - start_time).total_seconds():.0f} seconds')
print(f'\nCompleted at {end_time.strftime("%Y-%m-%d %H:%M:%S")} (UTC)\n')
